# Subnational Data Merge Pipeline (Step-by-Step)

This notebook breaks down the merge process into individual cells for debugging.

In [1]:
import os
import logging
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterstats import zonal_stats
import glob
import difflib
from shapely.geometry import Point
import itertools

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Base Data Directory
# User specified to use 'data' (relative to notebook root)
DATA_DIR = "data"

# Parameters
iso3_list = ['KEN', 'SOM', 'ETH']
boundary_dir = os.path.join(DATA_DIR, 'geoboundaries')

print("Environment ready.")

Environment ready.


## 1. Helper Functions

In [2]:
def load_canonical_boundaries(iso3_list, boundary_dir):
    gdfs = []
    # User provided paths:
    # /geoboundaries/gb_SOM_ADM2.geojson
    # /geoboundaries/gb_ETH_ADM2.geojson
    # /geoboundaries/gb_KEN_ADM2.geojson
    
    file_map = {
        'KEN': 'gb_KEN_ADM2.geojson',
        'SOM': 'gb_SOM_ADM2.geojson',
        'ETH': 'gb_ETH_ADM2.geojson'
    }
    
    for iso in iso3_list:
        filename = file_map.get(iso)
        path = os.path.join(boundary_dir, filename)
        
        if os.path.exists(path):
            try:
                gdf = gpd.read_file(path)
                
                # Standardize columns
                # Rename common variants to shapeName
                rename_map = {
                    'shapeName_ADM2': 'shapeName',
                    'ADM2_NAME': 'shapeName',
                    'admin2Name': 'shapeName',
                    'shapeName': 'shapeName' # Identity
                }
                gdf.rename(columns=rename_map, inplace=True)
                
                if 'shapeName' not in gdf.columns:
                    logger.warning(f"Could not identify ADM2 name column for {iso}. Available: {gdf.columns.tolist()}")
                    continue
                
                # FORCE shapeISO to be the COUNTRY ISO
                # User noted that native shapeISO contains admin2 codes (e.g. KEN-1-1)
                # We need it to be 'KEN', 'SOM', 'ETH' for filtering later
                gdf['shapeISO'] = iso
                
                # Select columns
                cols = ['shapeName', 'shapeISO', 'geometry']
                # Keep whatever is available
                valid_cols = [c for c in cols if c in gdf.columns]
                
                gdfs.append(gdf[valid_cols])
                logger.info(f"Loaded {iso} boundaries from {filename}: {len(gdf)} regions")
            except Exception as e:
                logger.error(f"Failed to load {path}: {e}")
        else:
            logger.error(f"Boundary file not found: {path}")
    
    if not gdfs:
        return gpd.GeoDataFrame()
    return pd.concat(gdfs, ignore_index=True)

def spatial_join_points(df, gdf, lon_col, lat_col):
    points = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326"
    )
    if gdf.crs != points.crs:
        gdf = gdf.to_crs(points.crs)
    joined = gpd.sjoin(points, gdf, how="left", predicate="within")
    joined.rename(columns={'shapeName': 'admin2_canonical'}, inplace=True)
    return joined

def fuzzy_match_names(series, choices, threshold=0.8):
    mapping = {}
    unique_names = series.dropna().unique()
    for name in unique_names:
        matches = difflib.get_close_matches(str(name), choices, n=1, cutoff=threshold)
        if matches:
            mapping[name] = matches[0]
    return mapping

def process_worldpop_population(country_gdf, raster_path, iso_code):
    logger.info(f"Processing WorldPop for {iso_code} with {len(country_gdf)} regions...")
    country_gdf = country_gdf[country_gdf['shapeISO'] == iso_code].copy()
    if country_gdf.empty:
        return pd.DataFrame()
    stats = zonal_stats(country_gdf, raster_path, stats="sum", all_touched=True)
    country_gdf['population'] = [s['sum'] for s in stats]
    return country_gdf[['shapeName', 'population']].rename(columns={'shapeName': 'admin2_canonical'})

def create_master_skeleton(years, months, admin_gdf):
    canonical_names = admin_gdf[['shapeName', 'shapeISO']].drop_duplicates()
    skeleton = []
    for year, month in itertools.product(years, months):
        temp = canonical_names.copy()
        temp['year'] = year
        temp['month'] = month
        skeleton.append(temp)
    master = pd.concat(skeleton, ignore_index=True)
    master.rename(columns={'shapeName': 'admin2', 'shapeISO': 'country_iso'}, inplace=True)
    return master

# Data Loading Functions
def load_price_data(data_dir):
    csv_path = os.path.join(data_dir, "worldbank_imputed_price_data/WLD_RTFP_mkt_2026-02-03.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(data_dir, "worldbank_imputed_price_data/WLD_RTFP_mkt_2026-01-13.csv")
    if not os.path.exists(csv_path):
        return pd.DataFrame()
    price = pd.read_csv(csv_path)
    target_countries = ['Kenya', 'Somalia']
    return price[price['country'].isin(target_countries)]

def load_crop_data(data_dir):
    agg_path = os.path.join(data_dir, "crop_mask/admin_mapped/admin_agg.parquet")
    if os.path.exists(agg_path):
        return pd.read_parquet(agg_path)
    else:
        return pd.DataFrame(columns=['shapeName_ADM2', 'value'])

def load_acled_data(data_dir):
    acled_path = os.path.join(data_dir, "raw/acled/Africa_aggregated_data_up_to-2026-01-03.xlsx")
    if not os.path.exists(acled_path):
        return pd.DataFrame()
    acled = pd.read_excel(acled_path)
    target_countries = ['Kenya', 'Ethiopia', 'Somalia']
    return acled[acled['COUNTRY'].isin(target_countries)]

## 2. Load Raw Data

In [3]:
print("Loading raw datasets...")
price_df = load_price_data(DATA_DIR)
crop_df = load_crop_data(DATA_DIR)
acled_df = load_acled_data(DATA_DIR)

print(f"Price Rows: {len(price_df)}")
print(f"Crop Rows: {len(crop_df)}")
print(f"ACLED Rows: {len(acled_df)}")

Loading raw datasets...
Price Rows: 62560
Crop Rows: 1656
ACLED Rows: 48606


## 3. Load Canonical Boundaries

In [5]:
logger.info(f"Loading canonical boundaries from {boundary_dir}...")
admin_gdf = load_canonical_boundaries(iso3_list, boundary_dir)
if not admin_gdf.empty:
    canonical_names = admin_gdf['shapeName'].unique().tolist()
    print(f"Loaded {len(admin_gdf)} total regions.")
    print(admin_gdf.head())
else:
    print("No boundaries loaded! Check file paths.")

2026-02-12 10:43:03,900 - INFO - Loading canonical boundaries from data/geoboundaries...
2026-02-12 10:43:04,109 - INFO - Loaded KEN boundaries from gb_KEN_ADM2.geojson: 290 regions
2026-02-12 10:43:04,119 - INFO - Loaded SOM boundaries from gb_SOM_ADM2.geojson: 118 regions
2026-02-12 10:43:04,140 - INFO - Loaded ETH boundaries from gb_ETH_ADM2.geojson: 74 regions


Loaded 482 total regions.
      shapeName shapeISO                                           geometry
0      Ainabkoi      KEN  POLYGON ((35.4633 0.5072, 35.46246 0.50691, 35...
1       Ainamoi      KEN  POLYGON ((35.32495 -0.25056, 35.3225 -0.24841,...
2         Aldai      KEN  POLYGON ((35.13034 0.1363, 35.12748 0.13635, 3...
3  Alego Usonga      KEN  POLYGON ((34.21848 0.16479, 34.21677 0.1634, 3...
4        Awendo      KEN  POLYGON ((34.62157 -0.9854, 34.61932 -0.9833, ...


In [6]:
admin_gdf

,shapeName,shapeISO,geometry
0,Ainabkoi,KEN,"POLYGON ((35.4633 0.5072, 35.46246 0.50691, 35..."
1,Ainamoi,KEN,"POLYGON ((35.32495 -0.25056, 35.3225 -0.24841,..."
2,Aldai,KEN,"POLYGON ((35.13034 0.1363, 35.12748 0.13635, 3..."
3,Alego Usonga,KEN,"POLYGON ((34.21848 0.16479, 34.21677 0.1634, 3..."
4,Awendo,KEN,"POLYGON ((34.62157 -0.9854, 34.61932 -0.9833, ..."
...,...,...,...
477,Zone 1,ETH,"POLYGON ((39.99832 11.61304, 39.99611 11.60491..."
478,Zone 2,ETH,"POLYGON ((39.74542 13.57146, 39.74033 13.5638,..."
479,Zone 3,ETH,"POLYGON ((40.09536 8.86322, 40.1059 8.87805, 4..."
480,Zone 4,ETH,"POLYGON ((39.99499 11.68196, 40.0074 11.70404,..."


## 4. Process Price Data

In [13]:
price_df

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,DATES,year,...,l_yogurt,c_yogurt,inflation_yogurt,trust_yogurt,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
220746,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-01-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.58,0.55,0.57,NaN,9.8
220747,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-02-01,2007,...,NaN,NaN,NaN,NaN,0.57,0.58,0.55,0.56,NaN,9.8
220748,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-03-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.57,0.54,0.54,NaN,9.8
220749,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-04-01,2007,...,NaN,NaN,NaN,NaN,0.54,0.56,0.53,0.56,NaN,9.8
220750,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-05-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.58,0.55,0.56,NaN,9.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
548681,SOM,Somalia,Market Average,Market Average,Market Average,NaN,NaN,gid_som_national_average,2025-10-01,2025,...,NaN,NaN,NaN,NaN,1.56,1.60,1.52,1.52,-1.88,9.9
548682,SOM,Somalia,Market Average,Market Average,Market Average,NaN,NaN,gid_som_national_average,2025-11-01,2025,...,NaN,NaN,NaN,NaN,1.52,1.56,1.49,1.52,-0.49,9.9
548683,SOM,Somalia,Market Average,Market Average,Market Average,NaN,NaN,gid_som_national_average,2025-12-01,2025,...,NaN,NaN,NaN,NaN,1.53,1.57,1.50,1.52,1.14,9.9
548684,SOM,Somalia,Market Average,Market Average,Market Average,NaN,NaN,gid_som_national_average,2026-01-01,2026,...,NaN,NaN,NaN,NaN,1.54,1.57,1.50,1.53,7.04,9.9


In [29]:
price_df_col =price_df[['year','month','ISO3', 'country','adm1_name','adm2_name','mkt_name','c_maize', 'lat', 'lon']]
price_df_col.dropna(subset=['lat', 'lon'], inplace=True)
price_df_col

/var/folders/t7/ldqv1xt97rs4jyjvhxf4shgw0000gn/T/ipykernel_6376/2576662558.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  price_df_col.dropna(subset=['lat', 'lon'], inplace=True)


,year,month,ISO3,country,adm1_name,adm2_name,mkt_name,c_maize,lat,lon
220746,2007,1,KEN,Kenya,Coast,Tana River,Adele Center,18.67,-0.47,39.61
220747,2007,2,KEN,Kenya,Coast,Tana River,Adele Center,16.70,-0.47,39.61
220748,2007,3,KEN,Kenya,Coast,Tana River,Adele Center,16.75,-0.47,39.61
220749,2007,4,KEN,Kenya,Coast,Tana River,Adele Center,15.76,-0.47,39.61
220750,2007,5,KEN,Kenya,Coast,Tana River,Adele Center,15.18,-0.47,39.61
...,...,...,...,...,...,...,...,...,...,...
548451,2025,10,SOM,Somalia,Bakool,Xudur,Xudur,22192.28,4.12,43.89
548452,2025,11,SOM,Somalia,Bakool,Xudur,Xudur,20875.00,4.12,43.89
548453,2025,12,SOM,Somalia,Bakool,Xudur,Xudur,25000.00,4.12,43.89
548454,2026,1,SOM,Somalia,Bakool,Xudur,Xudur,19954.62,4.12,43.89


In [31]:
logger.info("Standardizing Price Data via Spatial Join...")
if 'lat' in price_df_col.columns and 'lon' in price_df_col.columns:
    price_joined = spatial_join_points(price_df_col, admin_gdf, 'lon', 'lat')
    matched = price_joined['admin2_canonical'].notna().sum()
    total = len(price_joined)
    logger.info(f"Price spatial join: {matched}/{total} matched ({matched/total*100:.1f}%)")
# else:
#     logger.warning("Price data has no lat/lon columns. Falling back to fuzzy matching.")
#     price_mapping = fuzzy_match_names(price_df['adm2_name'], canonical_names)
#     price_joined = price_df.copy()
#     price_joined['admin2_canonical'] = price_joined['adm2_name'].map(price_mapping)


# KEEP ALL COMMODITIES that are present
# KEEP_COMMODITIES = ['sorghum', 'maize', 'rice', 'wheat', 'beans', 'millet', 'food_price_index']
# price_cols = [c for c in price_joined.columns
#                 if any(c == k or c.startswith(k + '_') or c.endswith('_' + k)
#                         or c.startswith('l_' + k) or c.startswith('o_' + k)
#                         or c.startswith('inflation_' + k) or c.startswith('trust_' + k)
#                         for k in KEEP_COMMODITIES)]
price_cols=['c_maize']
# Keep only numeric among selected
price_cols = price_joined[price_cols].select_dtypes(include=np.number).columns.tolist()
logger.info(f"Price columns kept: {len(price_cols)}")

price_agg = price_joined.groupby(
    ['year', 'month', 'admin2_canonical']
)[price_cols].mean().reset_index()

print("Price Aggregation Sample:")
print(len(price_joined))
print(price_agg.head())
print(len(price_agg))

2026-02-12 10:51:44,202 - INFO - Standardizing Price Data via Spatial Join...
2026-02-12 10:51:44,326 - INFO - Price spatial join: 61870/62330 matched (99.3%)
2026-02-12 10:51:44,328 - INFO - Price columns kept: 1


Price Aggregation Sample:
62330
   year  month admin2_canonical  c_maize
0  2007      1          ABUDWAQ  2941.28
1  2007      1            ADADO  2762.72
2  2007      1            ADALE  1953.80
3  2007      1            AFGOI  2084.28
4  2007      1          AFMADOW  1643.68
25760


In [42]:
# 1. Row 수 비교
print(f"Price 원본 개수: {len(price_df)}")
print(f"Join 이후 개수: {len(price_joined)}")
# 2. 매칭 실패한 데이터 확인
unmatched = price_joined[price_joined['admin2_canonical'].isna()]
print(f"매칭 실패(NaN) 개수: {len(unmatched)}")
if not unmatched.empty:
    print("\n매칭 실패한 데이터 샘플 (좌표 확인용):")
    # lat, lon 컬럼명은 실제 데이터에 맞게 조정 필요
    cols_to_show = ['country', 'mkt_name', 'lat', 'lon'] 
    # 만약 컬럼명이 다르면 아래처럼 확인해보세요:
    # print(price_joined.columns)
    print(unmatched[cols_to_show].head(10))

Price 원본 개수: 62560
Join 이후 개수: 62330
매칭 실패(NaN) 개수: 460

매칭 실패한 데이터 샘플 (좌표 확인용):
       country mkt_name   lat    lon
243286   Kenya   Kipini -2.53  40.53
243287   Kenya   Kipini -2.53  40.53
243288   Kenya   Kipini -2.53  40.53
243289   Kenya   Kipini -2.53  40.53
243290   Kenya   Kipini -2.53  40.53
243291   Kenya   Kipini -2.53  40.53
243292   Kenya   Kipini -2.53  40.53
243293   Kenya   Kipini -2.53  40.53
243294   Kenya   Kipini -2.53  40.53
243295   Kenya   Kipini -2.53  40.53


In [38]:
len(price_joined[price_joined["admin2_canonical"].isna()])

460

In [37]:
print("Price df len: ", len(price_df))
print(price_df_col.isna().sum())
print("Price joined len: ", len(price_joined))
print(price_joined.isna().sum())

Price df len:  62560
year         0
month        0
ISO3         0
country      0
adm1_name    0
adm2_name    0
mkt_name     0
c_maize      0
lat          0
lon          0
dtype: int64
Price joined len:  62330
year                  0
month                 0
ISO3                  0
country               0
adm1_name             0
adm2_name             0
mkt_name              0
c_maize               0
lat                   0
lon                   0
geometry              0
index_right         460
admin2_canonical    460
shapeISO            460
dtype: int64


## 5. Process Population Data

In [40]:
logger.info("Standardizing Population Data via WorldPop Zonal Stats...")
pop_dfs = []

# Map ISO to raster file
raster_map = {
    'KEN': 'ken_pop_2020_1km.tif',
    'SOM': 'som_pop_2020_1km.tif',
    'ETH': 'eth_pop_2020_1km.tif'
}

for iso in iso3_list:
    if iso not in raster_map:
        continue
        
    raster_path = Path(DATA_DIR) / 'population_worldpop' / raster_map[iso]
    
    if raster_path.exists():
        iso_pop = process_worldpop_population(admin_gdf, str(raster_path), iso)
        if not iso_pop.empty:
            pop_dfs.append(iso_pop)
    else:
        logger.warning(f"Population raster not found for {iso}: {raster_path}")

if pop_dfs:
    pop_agg = pd.concat(pop_dfs, ignore_index=True)
    pop_agg = pop_agg.groupby(['admin2_canonical'])['population'].sum().reset_index()
else:
    logger.warning("No population data processed.")
    pop_agg = pd.DataFrame(columns=['admin2_canonical', 'population'])

print("Population Aggregation Sample:")
print(pop_agg.head())
print(len(pop_agg))
print(pop_agg.isna().sum())

2026-02-12 11:00:22,529 - INFO - Standardizing Population Data via WorldPop Zonal Stats...
2026-02-12 11:00:22,529 - INFO - Processing WorldPop for KEN with 482 regions...
2026-02-12 11:00:23,397 - INFO - Processing WorldPop for SOM with 482 regions...
2026-02-12 11:00:23,640 - INFO - Processing WorldPop for ETH with 482 regions...


Population Aggregation Sample:
  admin2_canonical     population
0       ABDUL AZIZ  115039.851562
1          ABUDWAQ   93101.750000
2            ADADO  119249.250000
3            ADALE   62335.511719
4       ADEN YABAL   75302.695312
482
admin2_canonical    0
population          0
dtype: int64


## 6. Process Crop Data

In [41]:
logger.info("Standardizing Crop Data...")
crop_df_proc = crop_df.copy()
# Filter to target countries
if 'shapeISO_ADM0' in crop_df_proc.columns:
    crop_df_proc = crop_df_proc[crop_df_proc['shapeISO_ADM0'].isin(iso3_list)]
    logger.info(f"Crop data filtered to {iso3_list}: {len(crop_df_proc)} rows")

canonical_set = set(canonical_names)
crop_df_proc['admin2_canonical'] = crop_df_proc['shapeName_ADM2'].where(
    crop_df_proc['shapeName_ADM2'].isin(canonical_set)
)
unmatched_crop = crop_df_proc['admin2_canonical'].isna().sum()
if unmatched_crop > 0:
    logger.warning(f"Crop data: {unmatched_crop} rows with unmatched Admin2 names. Falling back to fuzzy match.")
    remaining_crop = crop_df_proc.loc[crop_df_proc['admin2_canonical'].isna(), 'shapeName_ADM2']
    crop_fallback = fuzzy_match_names(remaining_crop, canonical_names)
    crop_df_proc.loc[crop_df_proc['admin2_canonical'].isna(), 'admin2_canonical'] = remaining_crop.map(crop_fallback)

crop_agg = crop_df_proc.dropna(subset=['admin2_canonical']).groupby(
    ['admin2_canonical']
)['value'].mean().reset_index().rename(columns={'value': 'crop_cover_fraction'})

print("Crop Aggregation Sample:")
print(crop_agg.head())
print(len(crop_agg))
print(crop_agg.isna().sum())

2026-02-12 11:00:47,741 - INFO - Standardizing Crop Data...
2026-02-12 11:00:47,743 - INFO - Crop data filtered to ['KEN', 'SOM', 'ETH']: 454 rows


Crop Aggregation Sample:
  admin2_canonical  crop_cover_fraction
0          ABUDWAQ             4.000000
1            ADADO             6.117647
2            ADALE            39.122749
3       ADEN YABAL             5.155779
4            AFGOI            52.040225
454
admin2_canonical       0
crop_cover_fraction    0
dtype: int64


## 7. Process ACLED Data

In [44]:
acled_df.columns

Index(['WEEK', 'REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE',
       'EVENTS', 'FATALITIES', 'POPULATION_EXPOSURE', 'DISORDER_TYPE', 'ID',
       'CENTROID_LATITUDE', 'CENTROID_LONGITUDE'],
      dtype='object')

In [56]:
#Later work on the feature
acled_df.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
70832,1997-10-04,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,3,NaN,Political violence,896.0,8.9644,38.7756
70833,2001-11-24,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,1,NaN,Political violence,896.0,8.9644,38.7756
70834,2002-04-06,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,100,NaN,Political violence,896.0,8.9644,38.7756
70835,2011-03-19,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,0,NaN,Political violence,896.0,8.9644,38.7756
70836,2011-09-10,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,4,NaN,Political violence,896.0,8.9644,38.7756


In [45]:
logger.info("Joining ACLED Data via Spatial Join...")
acled_joined = spatial_join_points(acled_df, admin_gdf, 'CENTROID_LONGITUDE', 'CENTROID_LATITUDE')

acled_joined['year'] = pd.to_datetime(acled_joined['WEEK']).dt.year
acled_joined['month'] = pd.to_datetime(acled_joined['WEEK']).dt.month

acled_agg = acled_joined.dropna(subset=['admin2_canonical']).groupby(
    ['year', 'month', 'admin2_canonical']
).agg({
    'FATALITIES': 'sum',
    'EVENTS': 'count'
}).reset_index().rename(columns={'EVENTS': 'conflict_events', 'FATALITIES': 'conflict_fatalities'})

print("ACLED Aggregation Sample:")
print(acled_agg.head())

2026-02-12 11:05:49,350 - INFO - Joining ACLED Data via Spatial Join...


ACLED Aggregation Sample:
   year  month  admin2_canonical  conflict_fatalities  conflict_events
0  1997      1            Agnuak                    0                1
1  1997      1             Jomvu                    2                1
2  1997      1             Loima                    3                1
3  1997      1  Nakuru Town East                    1                2
4  1997      2           AFMADOW                    1                1


## 8. Final Merge

In [50]:
years = sorted(price_df['year'].unique())
months = sorted(price_df['month'].unique())

logger.info(f"Creating skeleton for {len(years)} years, {len(months)} months, {len(canonical_names)} regions...")
master = create_master_skeleton(years, months, admin_gdf)

logger.info("Merging all datasets...")

merged = pd.merge(master, price_agg,
                    left_on=['year', 'month', 'admin2'],
                    right_on=['year', 'month', 'admin2_canonical'], how='left')
# if 'admin2_canonical' in merged.columns:
    # merged.drop(columns=['admin2_canonical'], inplace=True)


2026-02-12 11:07:55,539 - INFO - Creating skeleton for 20 years, 12 months, 482 regions...
2026-02-12 11:07:55,595 - INFO - Merging all datasets...


In [ ]:

merged = pd.merge(merged, pop_agg,
                    left_on='admin2', right_on='admin2_canonical', how='left')
if 'admin2_canonical' in merged.columns:
    merged.drop(columns=['admin2_canonical'], inplace=True)

merged = pd.merge(merged, crop_agg,
                    left_on='admin2', right_on='admin2_canonical', how='left')
if 'admin2_canonical' in merged.columns:
    merged.drop(columns=['admin2_canonical'], inplace=True)

merged = pd.merge(merged, acled_agg,
                    left_on=['year', 'month', 'admin2'],
                    right_on=['year', 'month', 'admin2_canonical'], how='left')
if 'admin2_canonical' in merged.columns:
    merged.drop(columns=['admin2_canonical'], inplace=True)

merged['conflict_events'] = merged['conflict_events'].fillna(0)
merged['conflict_fatalities'] = merged['conflict_fatalities'].fillna(0)

logger.info(f"Final merged shape: {merged.shape}")
print(merged.head())